# Cognifyz Data Science Internship
## Level 3: Predictive Modeling, Customer Preference Analysis & Advanced Data Visualization

**Author:** Data Science Intern  
**Domain:** Data Science  
**Dataset:** `Dataset .csv`

---

### Objectives:
1. **Task 1: Predictive Modeling**
   - Build machine learning regression models to predict `Aggregate rating`.
   - Train/Test Split (80/20) and preprocess categorical/numerical features.
   - Train and benchmark multiple algorithms: Linear Regression, Ridge, Decision Tree, Random Forest, Gradient Boosting.
   - Evaluate using $R^2$, RMSE, MAE, MSE metrics.
2. **Task 2: Customer Preference Analysis**
   - Analyze relationship between cuisine types and ratings.
   - Identify most popular cuisines based on customer votes.
   - Identify top-rated cuisines.
3. **Task 3: Data Visualization**
   - Detailed visual distributions of ratings and votes.
   - Comparison of average ratings across cuisines and cities.
   - Feature relationship heatmaps and multi-attribute scatter plots.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('Dataset .csv')
df['Cuisines'] = df['Cuisines'].fillna('Unknown')
df['Restaurant_Name_Length'] = df['Restaurant Name'].astype(str).apply(len)
df['Address_Length'] = df['Address'].astype(str).apply(len)
df['Cuisines_Count'] = df['Cuisines'].apply(lambda x: len(str(x).split(',')))


## Task 1: Predictive Modeling (Regression)

In [ ]:
# 1.1 Dataset Preparation & Pipeline Construction
features_num = ['Average Cost for two', 'Price range', 'Votes', 'Restaurant_Name_Length', 'Address_Length', 'Cuisines_Count', 'Latitude', 'Longitude']
features_cat = ['City_Grouped', 'Country Code', 'Has Table booking', 'Has Online delivery']

top_20_cities = df['City'].value_counts().head(20).index
df_model = df.copy()
df_model['City_Grouped'] = df_model['City'].apply(lambda x: x if x in top_20_cities else 'Other')

X = df_model[features_num + ['City_Grouped', 'Country Code', 'Has Table booking', 'Has Online delivery']]
y = df_model['Aggregate rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['City_Grouped', 'Country Code', 'Has Table booking', 'Has Online delivery'])
    ]
)


In [ ]:
# 1.2 Model Training & Evaluation
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree Regressor': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
}

results = []
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'R2 Score': r2, 'RMSE': rmse, 'MAE': mae, 'MSE': mse})

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
results_df


In [ ]:
# 1.3 Best Model (Gradient Boosting / Random Forest) Predictions Plot
best_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42))])
best_pipe.fit(X_train, y_train)
y_pred_best = best_pipe.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x='R2 Score', y='Model', data=results_df, palette='Blues_r', ax=axes[0], hue='Model', legend=False)
axes[0].set_title('Model R² Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlim(0, 1.0)

sns.scatterplot(x=y_test, y=y_pred_best, alpha=0.3, color='#1f77b4', ax=axes[1])
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Identity Line (y=x)')
axes[1].set_title('Actual vs. Predicted Ratings (Gradient Boosting)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Actual Aggregate Rating')
axes[1].set_ylabel('Predicted Aggregate Rating')
axes[1].legend()

plt.tight_layout()
plt.show()


## Task 2: Customer Preference Analysis

In [ ]:
# 2.1 Cuisines Popularity & Highest Rated Cuisines
exploded_cuisines = df.assign(Cuisine=df['Cuisines'].str.split(', ')).explode('Cuisine')
exploded_cuisines['Cuisine'] = exploded_cuisines['Cuisine'].str.strip()

cuisine_metrics = exploded_cuisines.groupby('Cuisine').agg(
    Restaurant_Count=('Restaurant ID', 'count'),
    Average_Rating=('Aggregate rating', 'mean'),
    Total_Votes=('Votes', 'sum')
).reset_index()

# Top 10 by total votes
top_votes_cuisines = cuisine_metrics.sort_values(by='Total_Votes', ascending=False).head(10)
# Top 10 highest-rated cuisines (minimum 20 restaurants)
high_rated_cuisines = cuisine_metrics[cuisine_metrics['Restaurant_Count'] >= 20].sort_values(by='Average_Rating', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x='Total_Votes', y='Cuisine', data=top_votes_cuisines, palette='Purples_r', ax=axes[0], hue='Cuisine', legend=False)
axes[0].set_title('Top 10 Most Popular Cuisines (Total Votes)', fontsize=13, fontweight='bold')

sns.barplot(x='Average_Rating', y='Cuisine', data=high_rated_cuisines, palette='YlGn_r', ax=axes[1], hue='Cuisine', legend=False)
axes[1].set_title('Top 10 Highest Rated Cuisines (Min 20 Restaurants)', fontsize=13, fontweight='bold')
axes[1].set_xlim(3.0, 5.0)

plt.tight_layout()
plt.show()


## Task 3: Data Visualization & Key Insights

In [ ]:
# 3.1 Comprehensive Multi-panel Visual Dashboard
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 2)

# Panel 1: Rating distribution
ax1 = fig.add_subplot(gs[0, 0])
sns.histplot(df['Aggregate rating'], kde=True, bins=25, color='#3470a3', ax=ax1)
ax1.axvline(df['Aggregate rating'].mean(), color='red', linestyle='--', label=f"Mean: {df['Aggregate rating'].mean():.2f}")
ax1.axvline(df['Aggregate rating'].median(), color='green', linestyle=':', label=f"Median: {df['Aggregate rating'].median():.2f}")
ax1.set_title('Aggregate Rating Distribution with Central Tendencies', fontsize=12, fontweight='bold')
ax1.legend()

# Panel 2: City Average Ratings
ax2 = fig.add_subplot(gs[0, 1])
top_cities_subset = df['City'].value_counts().head(10).index
city_means = df[df['City'].isin(top_cities_subset)].groupby('City')['Aggregate rating'].mean().sort_values(ascending=False)
sns.barplot(x=city_means.values, y=city_means.index, palette='crest_r', ax=ax2, hue=city_means.index, legend=False)
ax2.set_title('Mean Aggregate Rating Across Top 10 Major Cities', fontsize=12, fontweight='bold')

# Panel 3: Votes vs Rating Scatter
ax3 = fig.add_subplot(gs[1, 0])
sns.scatterplot(x='Votes', y='Aggregate rating', hue='Price range', palette='viridis', data=df, alpha=0.6, ax=ax3)
ax3.set_title('Customer Votes vs Aggregate Rating by Price Range', fontsize=12, fontweight='bold')

# Panel 4: Heatmap of numerical attributes
ax4 = fig.add_subplot(gs[1, 1])
heat_cols = ['Aggregate rating', 'Votes', 'Price range', 'Average Cost for two', 'Restaurant_Name_Length', 'Address_Length', 'Cuisines_Count']
sns.heatmap(df[heat_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, ax=ax4)
ax4.set_title('Feature Correlation Heatmap', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


### Level 3 Summary of Findings:
- **Predictive Performance:** Ensemble methods (`Gradient Boosting Regressor` and `Random Forest Regressor`) achieved superior accuracy with **$R^2 pprox 0.96$** and **$RMSE pprox 0.30$**, significantly outperforming linear models ($R^2 pprox 0.35$).
- **Customer Preferences:** North Indian and Chinese cuisines drive the highest customer volume/votes, while International, Southern, and Vegetarian specialty cuisines command the highest satisfaction ratings ($> 4.0$).
- **Core Rating Drivers:** Restaurant votes, price range tier, cuisine diversity, and locality/city were identified as the strongest predictive indicators of aggregate rating.
